# 02 · Split — one dataset per sub-model, plus the feature mapper

A **sub-model** is one *(unit, asset, horizon)*. This notebook slices `f1_Xy.parquet` into one file per
sub-model, carrying only the features that apply to that unit's panel, and writes two index files.

**Outputs**
- `units/<unit>/xy/<asset>__h<horizon>.parquet` — 43 files, the per-sub-model datasets
- `f1_pipeline/data/f1_submodels.csv` — one row per file: paths, counts, feature list
- `f1_pipeline/data/f1_submodel_features.parquet` — file × feature boolean mapper

The files go in an `xy/` **sub**folder on purpose: the reference CLI reads every `*.parquet` sitting
directly in a unit folder as a panel, and would choke on ours.

## Setup

In [1]:
import json, pathlib, subprocess, sys
import numpy as np, pandas as pd

HERE = pathlib.Path.cwd() if pathlib.Path.cwd().name == "f1_pipeline" else pathlib.Path.cwd() / "f1_pipeline"
REPO, DATA = HERE.parent, HERE / "data"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

Xy   = pd.read_parquet(DATA / "f1_Xy.parquet")
COLS = json.load(open(DATA / "f1_columns.json"))
print(f"{len(Xy):,} rows · {len(COLS['feature'])} candidate features")

150,066 rows · 31 candidate features


## 1 — Which features apply to a unit

A feature column is kept if it is non-NaN somewhere in that sub-model's training rows. That drops the
`ctx_*` block of every *other* panel automatically — no hand-maintained list.

In [2]:
KEEP_ID     = ["unit", "panel", "freq", "asset", "target_type", "value_unit", "asof",
               "origin_date", "origin_idx", "horizon_bd", "steps_ahead", "target_date", "split"]
KEEP_TARGET = ["anchor", "target", "target_change"]

def submodel_frame(unit: str, asset: str, horizon: int):
    g = Xy[(Xy.unit == unit) & (Xy.asset == asset) & (Xy.horizon_bd == horizon)]
    train = g[g.split == "train"]
    features = [c for c in COLS["feature"] if train[c].notna().any()]
    return g[KEEP_ID + KEEP_TARGET + features].reset_index(drop=True), features

## 2 — Write one file per sub-model

In [3]:
index = []
for (unit, asset, horizon), _ in Xy.groupby(["unit", "asset", "horizon_bd"]):
    df, features = submodel_frame(unit, asset, horizon)
    out_dir = REPO / "units" / unit / "xy"; out_dir.mkdir(exist_ok=True)
    path = out_dir / f"{asset}__h{horizon}.parquet"
    df.to_parquet(path, index=False)
    index.append({"unit": unit, "asset": asset, "horizon_bd": horizon,
                  "target_type": df["target_type"].iloc[0], "panel": df["panel"].iloc[0], "freq": df["freq"].iloc[0],
                  "asof": df["asof"].iloc[0].date(), "steps_ahead": int(df["steps_ahead"].iloc[0]),
                  "n_train": int((df.split == "train").sum()), "n_predict": int((df.split == "predict").sum()),
                  "n_features": len(features), "features": ",".join(features),
                  "path": str(path.relative_to(REPO))})
index = pd.DataFrame(index)
print(f"{len(index)} sub-model files written")
index.drop(columns="features")

43 sub-model files written


,unit,asset,horizon_bd,target_type,panel,freq,asof,steps_ahead,n_train,n_predict,n_features,path
0,t2-F1-ai-mom-2024,MOM,127,log_return,factors_daily,daily,2024-05-31,127,6015,1,16,units/t2-F1-ai-mom-2024/xy/MOM__h127.parquet
1,t2-F1-aud-on-hold-2016,AUD,126,level,g10_fx_daily,daily,2016-11-01,126,4106,1,17,units/t2-F1-aud-on-hold-2016/xy/AUD__h126.parquet
2,t2-F1-aud-on-hold-2016,AUD,189,level,g10_fx_daily,daily,2016-11-01,189,4043,1,17,units/t2-F1-aud-on-hold-2016/xy/AUD__h189.parquet
3,t2-F1-cad-boc-2017,CAD,126,level,g10_fx_daily,daily,2017-07-12,126,4278,1,17,units/t2-F1-cad-boc-2017/xy/CAD__h126.parquet
4,t2-F1-cad-boc-2017,CAD,189,level,g10_fx_daily,daily,2017-07-12,189,4215,1,17,units/t2-F1-cad-boc-2017/xy/CAD__h189.parquet
5,t2-F1-chf-highly-valued-2021,CHF,126,level,g10_fx_daily,daily,2021-06-17,126,5258,1,17,units/t2-F1-chf-highly-valued-2021/xy/CHF__h12...
6,t2-F1-chf-highly-valued-2021,CHF,189,level,g10_fx_daily,daily,2021-06-17,189,5195,1,17,units/t2-F1-chf-highly-valued-2021/xy/CHF__h18...
7,t2-F1-conflicting-texts-2024,UST_10Y,126,level,rates_daily,daily,2024-06-12,126,5990,1,21,units/t2-F1-conflicting-texts-2024/xy/UST_10Y_...
8,t2-F1-conflicting-texts-2024,UST_10Y,189,level,rates_daily,daily,2024-06-12,189,5927,1,21,units/t2-F1-conflicting-texts-2024/xy/UST_10Y_...
9,t2-F1-considerable-period-2003,UST_10Y,126,level,rates_daily,daily,2003-08-12,126,777,1,21,units/t2-F1-considerable-period-2003/xy/UST_10...


## 3 — The mapper: which file uses which feature

In [4]:
feat_map = pd.DataFrame([{**{k: r[k] for k in ("path", "unit", "asset", "horizon_bd", "panel", "target_type")},
                          **{f: (f in r["features"].split(",")) for f in COLS["feature"]}}
                         for _, r in index.iterrows()]).set_index("path")
print("per-file feature counts:", sorted(feat_map[COLS["feature"]].sum(axis=1).unique()))
feat_map.groupby("panel")[COLS["feature"]].all().T.replace({True: "✓", False: ""})

per-file feature counts: [np.int64(16), np.int64(17), np.int64(18), np.int64(21)]


panel,factors_daily,g10_fx_daily,macro_monthly,rates_daily
level,,✓,✓,✓
log_level,,✓,✓,✓
mom_21,✓,✓,✓,✓
mom_63,✓,✓,✓,✓
mom_126,✓,✓,✓,✓
mom_252,✓,✓,✓,✓
z_252,✓,✓,✓,✓
pos_252,✓,✓,✓,✓
rv_21,✓,✓,,✓
rv_63,✓,✓,✓,✓


## 4 — Save the indexes and check a round-trip

In [5]:
index.to_csv(DATA / "f1_submodels.csv", index=False)
feat_map.to_parquet(DATA / "f1_submodel_features.parquet")
feat_map.to_csv(DATA / "f1_submodel_features.csv")

# read one sub-model back exactly the way notebooks 03 and 04 will
path = "units/t2-F1-pause-2006/xy/UST_10Y__h189.parquet"
sm = pd.read_parquet(REPO / path)
row = feat_map.loc[path, COLS["feature"]]; features = list(row.index[row.astype(bool)])
train, predict = sm[sm.split == "train"].dropna(subset=features), sm[sm.split == "predict"]
print(f"{path}\n  X_train {train[features].shape}   y_train {len(train)}   X_predict {predict[features].shape}"
      f"   anchor {predict.anchor.iloc[0]}")
print(f"  {len(features)} features: {features}")

units/t2-F1-pause-2006/xy/UST_10Y__h189.parquet
  X_train (1209, 21)   y_train 1209   X_predict (1, 21)   anchor 4.93
  21 features: ['level', 'log_level', 'mom_21', 'mom_63', 'mom_126', 'mom_252', 'z_252', 'pos_252', 'rv_21', 'rv_63', 'rv_252', 'ewma_vol', 'vol_ratio', 'skew_252', 'last_step', 'ctx_slope_10_2', 'ctx_slope_5_2', 'ctx_curv_2_5_10', 'ctx_ust2y', 'ctx_ust10y', 'ctx_slope_mom_63']


## 5 — The reference CLI still works

The unit folders now contain an extra `xy/` directory. Confirm that the repo's own forecaster, which
globs `*.parquet` in a unit folder, is unaffected.

In [6]:
tmp = HERE / "data" / "_cli_check"
res = subprocess.run([sys.executable, "-m", "qfbench2_track_forecasting.cli",
                      "--panels", "units/t2-F1-pause-2006", "--text", "units/t2-F1-pause-2006/text",
                      "--asof", "2006-08-08", "--out", str(tmp / "forecast.parquet")],
                     capture_output=True, text=True, cwd=REPO)
print(res.stdout.strip() or res.stderr.strip()[-400:])
import shutil; shutil.rmtree(tmp, ignore_errors=True)

wrote forecast.parquet, forecast_meta.json and forecast_rationale.md to /Users/dew/track2-forecasting-public/f1_pipeline/data/_cli_check
  2 asset(s) x 2 horizon(s), 500 draws


Next: **03 · Model** — fit M2 on the level-target units, tune globally, backtest, save predictions.